In [1]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/data
!unzip -q /content/drive/MyDrive/dental/dental_crops.zip -d /content/data
!ls /content/data/Dental_Radiography

Mounted at /content/drive
test  train  valid


In [2]:
import torch, torch.nn as nn, numpy as np, json
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import classification_report, f1_score

ROOT = "/content/data/Dental_Radiography"
dev = "cuda"
norm = transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
tf_train = transforms.Compose([transforms.Grayscale(3), transforms.Resize(128),
    transforms.RandomRotation(8), transforms.ColorJitter(0.2,0.2),
    transforms.ToTensor(), norm])
tf_eval = transforms.Compose([transforms.Grayscale(3), transforms.Resize(128),
    transforms.ToTensor(), norm])

ds_tr = datasets.ImageFolder(f"{ROOT}/train", tf_train)
ds_va = datasets.ImageFolder(f"{ROOT}/valid", tf_eval)
ds_te = datasets.ImageFolder(f"{ROOT}/test",  tf_eval)
classes = ds_tr.classes
print(classes)

counts = np.bincount(ds_tr.targets)
w = (1.0 / counts)[ds_tr.targets]
sampler = WeightedRandomSampler(w, num_samples=len(w), replacement=True)
dl_tr = DataLoader(ds_tr, batch_size=128, sampler=sampler, num_workers=2)
dl_va = DataLoader(ds_va, batch_size=256, num_workers=2)
dl_te = DataLoader(ds_te, batch_size=256, num_workers=2)

['Cavity', 'Fillings', 'Impacted Tooth', 'Implant', 'Normal']


In [4]:
model = models.resnet18(weights="IMAGENET1K_V1")
model.fc = nn.Linear(model.fc.in_features, len(classes))
model = model.to(dev)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
EPOCHS = 8
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=1e-3,
        steps_per_epoch=len(dl_tr), epochs=EPOCHS)
loss_fn = nn.CrossEntropyLoss()

def predict(dl):
    model.eval(); P=[]; Y=[]
    with torch.no_grad():
        for x,y in dl:
            P += model(x.to(dev)).argmax(1).cpu().tolist(); Y += y.tolist()
    return Y, P

best = 0
for ep in range(EPOCHS):
    model.train()
    for x,y in dl_tr:
        x,y = x.to(dev), y.to(dev)
        opt.zero_grad(); loss = loss_fn(model(x), y); loss.backward(); opt.step(); sched.step()
    Y,P = predict(dl_va)
    f1 = f1_score(Y,P,average="macro")
    print(f"epoch {ep+1}/{EPOCHS}  val macro-F1 {f1:.3f}")
    if f1 > best:
        best = f1
        torch.save(model.state_dict(), "/content/tooth_classifier.pt")

epoch 1/8  val macro-F1 0.618
epoch 2/8  val macro-F1 0.506
epoch 3/8  val macro-F1 0.631
epoch 4/8  val macro-F1 0.580
epoch 5/8  val macro-F1 0.684
epoch 6/8  val macro-F1 0.745
epoch 7/8  val macro-F1 0.747
epoch 8/8  val macro-F1 0.743


In [5]:
model.load_state_dict(torch.load("/content/tooth_classifier.pt"))
Y,P = predict(dl_te)
print(classification_report(Y,P,target_names=classes,digits=3))
json.dump(classes, open("/content/classes.json","w"))
!cp /content/tooth_classifier.pt /content/classes.json /content/drive/MyDrive/dental/

                precision    recall  f1-score   support

        Cavity      0.400     0.273     0.324        22
      Fillings      0.878     0.911     0.894       315
Impacted Tooth      0.643     0.562     0.600        32
       Implant      0.922     0.904     0.913       104
        Normal      0.955     0.956     0.955      1176

      accuracy                          0.927      1649
     macro avg      0.759     0.721     0.737      1649
  weighted avg      0.925     0.927     0.926      1649

